In [1]:
%matplotlib qt

In [2]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import scipy as sp
import cartopy.crs as ccrs
from numba import njit, prange
from tqdm import tqdm
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV, LeaveOneOut

In [3]:
df = pd.read_csv('sdss_cutout.csv')

In [4]:
df

,RA,DEC,Z,phot_u,phot_g,phot_r,phot_i,phot_z
0,130.089951,52.178097,0.097629,19.854843,18.159964,17.345013,16.930433,16.620535
1,130.173909,52.557767,0.067737,19.250399,17.239248,16.517265,16.083832,15.822161
2,130.081237,52.668188,0.063350,18.680342,17.206959,16.697826,16.362032,16.173033
3,130.291706,52.572373,0.065425,19.726742,17.791235,16.906610,16.495539,16.182104
4,130.013537,52.766588,0.123360,19.386818,17.534359,16.450377,15.943369,15.545097
...,...,...,...,...,...,...,...,...
319953,149.580633,47.033396,0.390038,23.223936,19.346468,17.534588,16.857544,16.351515
319954,150.261929,46.827647,0.204686,20.360432,18.511255,17.276648,16.812563,16.496658
319955,149.937963,46.482358,0.102099,19.087252,17.185118,16.237055,15.844631,15.480799
319956,150.023597,46.260926,0.069142,19.072407,17.949339,17.497790,17.227016,17.132250


In [5]:
data_cut = df[(df['Z'] > 0.09) & (df['Z'] < 0.1)].copy()

In [6]:
data_cut

,RA,DEC,Z,phot_u,phot_g,phot_r,phot_i,phot_z
0,130.089951,52.178097,0.097629,19.854843,18.159964,17.345013,16.930433,16.620535
13,130.663027,50.071896,0.095350,20.303741,18.293287,17.320787,16.889143,16.556585
55,130.261304,51.800044,0.092331,19.434448,17.842188,17.030638,16.609392,16.311544
56,130.076575,51.874619,0.097099,19.325104,17.350431,16.404455,15.959698,15.633738
57,130.009586,51.810246,0.097393,19.497576,17.759668,16.880379,16.463339,16.130400
...,...,...,...,...,...,...,...,...
319768,147.928786,45.079830,0.093443,19.471977,18.266950,17.705355,17.348024,17.118505
319772,147.753756,45.591582,0.095352,19.610746,17.991503,17.204578,16.799559,16.477268
319805,147.177855,45.635451,0.099245,19.104990,17.853140,16.089037,15.667090,15.316325
319816,147.104076,45.728887,0.096366,19.538519,18.621540,17.144320,16.678907,16.316566


In [7]:
data_cut['u-r'] = data_cut['phot_u'] - data_cut['phot_r']

In [8]:
red = data_cut[data_cut['u-r'] > 2.3]
blue = data_cut[data_cut['u-r'] <= 2.3]

In [9]:
fig, ax = plt.subplots(1, 2, figsize=(10, 15), subplot_kw={'projection': ccrs.Mollweide(180)},
                        layout='constrained')

ax[0].gridlines()
ax[0].scatter(red['RA'], red['DEC'], s=0.5, alpha=0.1, transform=ccrs.PlateCarree(), color='red')

ax[1].gridlines()
ax[1].scatter(blue['RA'], blue['DEC'], s=0.5, alpha=0.1, transform=ccrs.PlateCarree(), color='blue')


In [10]:
red_np = np.array([red['RA'].values, red['DEC'].values]).T * np.pi/180.

In [17]:
def uniform_grid(data, n=(1000, 1000)):
    x_min, y_min = np.min(data, axis=0)
    x_max, y_max = np.max(data, axis=0)

    x_sample = np.linspace(x_min, x_max, n[0])
    y_sample = np.linspace(y_min, y_max, n[1])

    res = np.meshgrid(x_sample, y_sample)
    return res

In [18]:
X, Y = uniform_grid(red_np, (50, 50))

In [42]:
xy = np.vstack([X.ravel(), Y.ravel()]).T

In [143]:
kde = KernelDensity(kernel='gaussian', bandwidth=0.0879).fit(red_np)

In [149]:
density_map = np.exp(kde.score_samples(xy)).reshape(X.shape)

In [154]:
kde2 = KernelDensity(kernel='gaussian', bandwidth=0.20).fit(red_np)
density_map2 = np.exp(kde2.score_samples(xy)).reshape(X.shape)

In [156]:
figure, ax = plt.subplots(1, 2)
ax[0].contourf(X, Y, density_map2)
ax[0].scatter(red_np[:,0], red_np[:,1], s=0.2, alpha=0.8, color='red')
ax[0].set_title('5-fold cross-validation')
ax[1].contourf(X, Y, density_map)
ax[1].scatter(red_np[:,0], red_np[:,1], s=0.2, alpha=0.8, color='red')
ax[1].set_title('Silverman')

Text(0.5, 1.0, 'Silverman')

In [147]:
bandwidths = np.geomspace(0.05, 2.0, 30)
grid = GridSearchCV(KernelDensity(kernel='gaussian'),
{'bandwidth': bandwidths}, cv=10, n_jobs=-1)
grid.fit(red_np) # Xdata has shape (N,2) f2.0or 2D
h_opt = grid.best_params_['bandwidth']

In [132]:
h_opt

np.float64(0.31622776601683794)

In [133]:
bandwidths

array([0.05      , 0.31622777, 2.        ])

In [134]:
plt.figure()
plt.plot(bandwidths, grid.cv_results_['mean_test_score'])
plt.xscale('log')

In [139]:
bandwidths = np.geomspace(0.005, 0.05, 3)
grid2 = GridSearchCV(KernelDensity(kernel='gaussian'),
{'bandwidth': bandwidths}, cv=LeaveOneOut(), n_jobs=-1)
grid2.fit(red_np) # Xdata has shape (N,2) f2.0or 2D
h_opt = grid2.best_params_['bandwidth']

In [140]:
h_opt

np.float64(0.005)

In [142]:
plt.figure()
plt.plot(bandwidths, grid2.cv_results_['mean_test_score'])
plt.xscale('log')